In [7]:
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import evaluate
import pandas as pd
import numpy as np
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split

In [8]:
seed = 6

In [9]:
df = pd.read_csv('../zorgdata/df_rapportages.csv')

# Verdeel de data in train- en testsets
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=seed)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=seed)


In [10]:
# Converteer naar Hugging Face 'Dataset' objecten
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

# Creëer een DatasetDict met de train-, validatie- en testsets
dataset = DatasetDict({
    'train': train_dataset, 
    'validation': valid_dataset, 
    'test': test_dataset
})


Aanpassen: Ander model nemen. Bert? Omzetten zoals eerder gedaan... dan peften

In [11]:
# Laad en configureer het model
model_name = 't5-small'
# model_name = 'google/mt5-small'
# model_name = 'google/flan-t5-small'
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.float32)
tokenizer = AutoTokenizer.from_pretrained(model_name)

original_model.to('cpu')

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [14]:
def tokenize_function(to_tokenize):
    prompt = [pt for pt in to_tokenize["rapportage"]]
    to_tokenize['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True, return_tensors="pt").input_ids
    to_tokenize['labels'] = tokenizer(to_tokenize["onrustscore"], padding="max_length", truncation=True, return_tensors="pt").input_ids
    
    return to_tokenize

vb = dataset['test'][1]
tokenize_function(vb)

print(tokenize_function(df.iloc[1]))


ValueError: text input must of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).